In [1]:
# --- Importação de Bibliotecas Padrão e de Terceiros ---
import os
import sys


# --- Configuração do Caminho do Projeto para Importações Locais ---
current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

import pandas as pd
import numpy as np
import joblib
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import IsolationForest

In [2]:
df = pd.read_csv(r"C:\repositorio\public-data-analysis-anomaly-detection-ml\data\processed\notas_fiscais.csv",sep=";",encoding="utf-8")

# Preparação dos Dados para o Isolation Forest

In [3]:
# --- 1. Tratamento de valores faltantes ---
# A coluna 'EVENTOS' possui 160 valores nulos. Preencher com 'Sem Evento'.
df["EVENTOS"] = df["EVENTOS"].fillna("Sem Evento")
print("Valores nulos após tratamento (EVENTOS):\n")
print(df.isnull().sum().to_markdown(numalign="left", stralign="left"))

# --- 2. Feature Engineering ---
# Converter a coluna 'DATA' para datetime
df["DATA"] = pd.to_datetime(df["DATA"])

Valores nulos após tratamento (EVENTOS):

|               | 0   |
|:--------------|:----|
| ID            | 0   |
| ORGAO         | 0   |
| FORNECEDOR    | 0   |
| CNPJ          | 0   |
| MUNICIPIO     | 0   |
| VALOR_NF      | 0   |
| ITENS         | 0   |
| TIPO_EVENTO   | 0   |
| DATA          | 0   |
| EVENTOS       | 0   |
| CHAVE_NF      | 0   |
| MUNICIPIO_MUN | 0   |
| UF            | 0   |
| POPULACAO     | 0   |
| COD_IBGE      | 0   |
| ANO_MES       | 0   |


In [ ]:
# Extrair features temporais
df["ANO"] = df["DATA"].dt.year
df["MES"] = df["DATA"].dt.month
df["DIA_SEMANA"] = df["DATA"].dt.dayofweek  # 0=Segunda, 6=Domingo
df["DIA_MES"] = df["DATA"].dt.day
df = df.drop('ITENS', axis=1)
df = df.drop('EVENTOS', axis=1)

# Criar feature de valor por população (VALOR_NF_POR_POPULACAO)
# Tratar casos onde POPULACAO pode ser zero para evitar divisão por zero
df["VALOR_NF_POR_POPULACAO"] = df["VALOR_NF"] / df["POPULACAO"].replace(0, np.nan) # Substituir 0 por NaN para depois preencher ou dropar
df["VALOR_NF_POR_POPULACAO"] = df["VALOR_NF_POR_POPULACAO"].fillna(0) # Preencher NaNs resultantes da divisão por zero com 0


In [21]:
print("\n### Primeiras 5 linhas do dataset com novas features:\n")
df.head()


### Primeiras 5 linhas do dataset com novas features:



,ID,ORGAO,FORNECEDOR,CNPJ,MUNICIPIO,VALOR_NF,TIPO_EVENTO,DATA,CHAVE_NF,MUNICIPIO_MUN,UF,POPULACAO,COD_IBGE,ANO_MES,ANO,MES,DIA_SEMANA,DIA_MES,VALOR_NF_POR_POPULACAO
0,1950,Ministério da Saúde - Unidades com vínculo direto,RCA PRODUTOS E SERVICOS LTDA,69207850000161,SANTA BARBARA D'OESTE,16004.55,Autorização de Uso,2021-11-01,35211169207850000161550010000008501783475076,SANTA BARBARA D'OESTE,SP,189.338,3545803,2021-11,2021,11,0,1,84.528990
1,1967,Ministério da Saúde - Unidades com vínculo direto,ATHOS RIO PRODUTOS MEDICOS HOSPITALARES LTDA,31912939000156,MESQUITA,3400.00,Autorização de Uso,2021-11-05,33211131912939000156550010000008061429011828,MESQUITA,MG,5.038,3141702,2021-11,2021,11,4,5,674.870981
2,1967,Ministério da Saúde - Unidades com vínculo direto,ATHOS RIO PRODUTOS MEDICOS HOSPITALARES LTDA,31912939000156,MESQUITA,3400.00,Autorização de Uso,2021-11-05,33211131912939000156550010000008061429011828,MESQUITA,RJ,178.803,3302858,2021-11,2021,11,4,5,19.015341
3,1984,Ministério da Saúde - Unidades com vínculo direto,ANJOMEDI DISTRIBUIDORA DE MEDICAMENTOS LTDA,31151224000128,ERECHIM,450.00,Autorização de Uso,2021-11-01,43211131151224000128550010000064451579723624,ERECHIM,RS,109.497,4307005,2021-11,2021,11,0,1,4.109702
4,2008,Ministério da Saúde - Unidades com vínculo direto,WALTER NICKHORN E CIA LTDA,88827019000157,PALMEIRA DAS MISSOES,6011.82,Autorização de Uso,2021-11-10,43211188827019000157550010000984061981262807,PALMEIRA DAS MISSOES,RS,34.233,4313706,2021-11,2021,11,2,10,175.614758


In [5]:
# --- 3. Normalização/Padronização e Codificação de Variáveis Categóricas ---
# Identificar colunas numéricas e categóricas para o pré-processamento
numeric_features = ["VALOR_NF", "POPULACAO", "VALOR_NF_POR_POPULACAO", "ANO", "MES", "DIA_SEMANA", "DIA_MES"]
# Colunas categóricas para One-Hot Encoding. Excluindo 'ID', 'CNPJ', 'CHAVE_NF' pois são identificadores únicos e 'DATA', 'ANO_MES' (já extraídas).
categorical_features = ["FORNECEDOR", "MUNICIPIO", "MUNICIPIO_MUN", "UF"]

In [6]:
# Criar pré-processador usando ColumnTransformer
# Para colunas numéricas, usar StandardScaler
# Para colunas categóricas, usar OneHotEncoder (handle_unknown='ignore' para evitar erros com categorias novas)
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])

In [7]:
# Aplicar o pré-processamento e converter a matriz esparsa para densa
X_processed = preprocessor.fit_transform(df).toarray()

# Obter nomes das colunas após One-Hot Encoding
onehot_feature_names = preprocessor.named_transformers_["cat"].get_feature_names_out(categorical_features)
processed_feature_names = numeric_features + list(onehot_feature_names)

In [8]:
df_processed = pd.DataFrame(X_processed, columns=processed_feature_names)

In [9]:
df_original_for_output = df.copy()

# Iniciando o Machine Learning

In [10]:
# Garantir que ambos os dataframes tenham o mesmo número de linhas
if df_processed.shape[0] != df_original_for_output.shape[0]:
    print("ERRO: O número de linhas entre o dataframe processado e o original não corresponde.")
    # Uma possível causa é a remoção de linhas devido a valores nulos em alguma etapa.
    exit()

In [11]:
# --- Treinar e aplicar o modelo Isolation Forest ---
# Definir o modelo Isolation Forest
contamination_rate = 0.05 # 5% de anomalias esperadas

print(f"Treinando Isolation Forest com contamination_rate = {contamination_rate}")

Treinando Isolation Forest com contamination_rate = 0.05


In [12]:
model = IsolationForest(n_estimators=100, contamination=contamination_rate, random_state=42, n_jobs=-1)

# Treinar o modelo
model.fit(df_processed)

,n_estimators,100
,max_samples,'auto'
,contamination,0.05
,max_features,1.0
,bootstrap,False
,n_jobs,-1
,random_state,42
,verbose,0
,warm_start,False


In [13]:
# Prever anomalias (-1 para anomalias, 1 para inliers)
anomaly_predictions = model.predict(df_processed)

In [14]:
# Obter os scores de decisão (quanto menor, mais anômalo)
anomaly_scores = model.decision_function(df_processed)

In [15]:
# Adicionar as previsões de anomalia e os scores de decisão ao DataFrame original
df_original_for_output["ANOMALY_PREDICTION"] = anomaly_predictions
df_original_for_output["ANOMALY_SCORE"] = anomaly_scores

In [16]:
# Mapear as previsões para "Sim" ou "Não"
df_original_for_output["POSSIVEL_ANOMALIA"] = df_original_for_output["ANOMALY_PREDICTION"].apply(lambda x: "Sim" if x == -1 else "Não")

print("\n### Contagem de Anomalias Detectadas:\n")
df_original_for_output["POSSIVEL_ANOMALIA"].value_counts()


### Contagem de Anomalias Detectadas:



POSSIVEL_ANOMALIA
Não    267933
Sim     14102
Name: count, dtype: int64

In [17]:
print("\n### Primeiras 5 linhas com rótulo de anomalia:\n")
df_original_for_output[df_original_for_output["POSSIVEL_ANOMALIA"] == "Sim"].head()


### Primeiras 5 linhas com rótulo de anomalia:



,ID,ORGAO,FORNECEDOR,CNPJ,MUNICIPIO,VALOR_NF,TIPO_EVENTO,DATA,CHAVE_NF,MUNICIPIO_MUN,...,COD_IBGE,ANO_MES,ANO,MES,DIA_SEMANA,DIA_MES,VALOR_NF_POR_POPULACAO,ANOMALY_PREDICTION,ANOMALY_SCORE,POSSIVEL_ANOMALIA
16,2355,Ministério da Saúde - Unidades com vínculo direto,BLAU FARMACEUTICA S.A.,58430828000160,COTIA,240.00,Carta de correção,2021-10-28,35211058430828000160550010002022671964431820,COTIA,...,3513009,2021-10,2021,10,3,28,0.836225,-1,-0.000187,Sim
23,2552,Ministério da Saúde - Unidades com vínculo direto,WHITE MARTINS GASES INDUSTRIAIS LTDA,35820448000721,DUQUE DE CAXIAS,320.00,Autorização de Uso,2021-11-11,33211135820448000721550720000205241859160128,DUQUE DE CAXIAS,...,3301702,2021-11,2021,11,3,11,0.369367,-1,-0.000772,Sim
28,3440,Ministério da Saúde - Unidades com vínculo direto,D E CERUTTI & CIA LTDA,2896716000144,PRIMAVERA DO LESTE,289.27,Autorização de Uso,2021-11-20,51211102896716000144550010003074271219172753,PRIMAVERA DO LESTE,...,5107040,2021-11,2021,11,5,20,3.112874,-1,-0.002023,Sim
30,3544,Ministério da Saúde - Unidades com vínculo direto,WHITE MARTINS GASES INDUSTRIAIS LTDA,35820448000721,DUQUE DE CAXIAS,1114.46,Autorização de Uso,2021-11-24,33211135820448000721552100000172061860575716,DUQUE DE CAXIAS,...,3301702,2021-11,2021,11,2,24,1.286390,-1,-0.001086,Sim
36,3756,Ministério da Saúde - Unidades com vínculo direto,WHITE MARTINS GASES INDUSTRIAIS LTDA,35820448000721,DUQUE DE CAXIAS,213.75,Autorização de Uso,2021-11-26,33211135820448000721550300000295631860907490,DUQUE DE CAXIAS,...,3301702,2021-11,2021,11,4,26,0.246726,-1,-0.001273,Sim


In [18]:
# Definir diretório de saída (pasta processed dentro de data)
output_dir = os.path.join(project_root, "data", "processed_isolation_forest")

# Salvar o dataset com as anomalias detectadas para a próxima fase
df_original_for_output.to_csv(os.path.join(output_dir,"notas_fiscais_com_anomalias.csv"), index=False, sep=";", encoding="utf-8")
print("Dataset com anomalias salvo como notas_fiscais_com_anomalias.csv")

Dataset com anomalias salvo como notas_fiscais_com_anomalias.csv
